<h1 style="text-align: center;">HEISENBERG-BASED FAULT LOCALIZATION (HBFL)</h1>

<h2 style="text-align: center;">Download Initial Dependencies</h2>

In [8]:
import numpy as np
import pandas as pd
import json
import math
from tqdm import tqdm
from qiskit.quantum_info import SparsePauliOp, Operator, Pauli, Clifford
import qiskit.qasm2 as q2
from qiskit import QuantumCircuit
from IPython.display import display

from heisenbergUtilities import *

In [9]:
# from qiskit_aer import AerSimulator
# simulator = AerSimulator()

# print(simulator.available_devices())

<h2 style="text-align: center;">CONTROL PANEL</h2>

In [10]:
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#-------------------------------------------------------------------------GENERAL PARAMETERS---------------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
ALG_NAME = "qpeexact"
ORIGINAL_QASM = f"{ALG_NAME}_5_qubits.qasm"                                                 #| The original algorithm QASM circuit file name.                           
ORIGINAL_QASM_FOLDER = "SBFL_circuits/MQTBench/"                                            #| The folder containing the original QASM circuit file.                    
QASM_MUTANT_FOLDER = f"SBFL_circuits/QMutBenchMutants/Mutants_{ALG_NAME}_5_qubits/"         #| The folder containing all mutants related to the original QASM circuit.  
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#-------------------------------------------------------------------------SB-QOPS PARAMETERS---------------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
RUNS = 10                                                                                   #| Number of times SB-QOPS will run to find a failing or passing test case.            
SHOTS = 20000                                                                               #| Number of shots for SB-QOPS to use to create the compact program specification.     
EPOCH = 80                                                                                  #| Number of epochs SB-QOPS will run to navigate the search space of test cases.       
SBQOPS_TOL = 0.1                                                                            #| Tolerance value SB-QOPS uses to determine if a test case is failing or passing.     
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#---------------------------------------------------------------------HEISENBERG SBFL PARAMETERS-----------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
LAMBDA_PHASE = 2                                                                        #| The hyperparameter used to weight the phase change of the Pauli propagation
LAMBDA_CHANGE = 1                                                                         #| The hyperparameter used to weight the change of string of the Pauli propagation
LAMBDA_SIMILARITY = 1                                                                       #| The hyperparameter used to weight the similarity difference between current and target Pauli
ATOL = 1e-4                                                                                 #| The tolerance value used to prune negligible magnitudes during Pauli propagation.   
MAX_TERMS = 100                                                                             #| The maximum number of terms to keep during Pauli propagation. If None, use all.     
SEARCH_STEP = 3                                                                             #| The search step size used during Pauli propagation. If None, pauli-prop uses 4.     
VERBOSE = 1                                                                                 #| A boolean value to determine if the program should print out detailed information.  
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#


Generate the linked list of all operations that take place in the inverse circuit

In [11]:
"""
LinkedListNode: A class that holds information related to a gate in a quantum circuit. 

PROPERTIES:
    - value (String): The name of the quantum gate
    - depth (int): The depth of the gate in the circuit
    - count (Int): The probability that the gate will meaningfully change the Pauli string.
    - next (LinkedListNode): The next gate in the circuit

"""

class LinkedListNode:
    def __init__(self, name="Initial", depth=0, count=0,  next=None):
        self.value = name
        self.depth = depth
        self.next = next
        self.count = count

class LinkedList:
    def __init__(self):
        self.head = None

    def append(self, name, depth):
        new_node = LinkedListNode(name, depth)
        if not self.head:
            self.head = new_node
            return
        last_node = self.head
        while last_node.next:
            last_node = last_node.next
        last_node.next = new_node

    def mark(self, depth):
        current_node = self.head
        while current_node:
            if current_node.depth == depth:
                current_node.count += 1
                return
            current_node = current_node.next

    def reset(self):
        current_node = self.head
        while current_node:
            current_node.count = 0
            current_node = current_node.next

<h3>Download MQT Benchmark circuits.</h3>

We'll start with DJ, RealAmpRandom, and GHZ since SB-QOPS caught those mutants 100% of the time for 5 qubit circuits



<h3>Generate mutants.</h3>

We'll start with mutants that add an unnecessary gate, then add mutants that replace a certain gate later.

The mutants were downloaded from QMutBench, choosing specifically mutants that added a gate anywhere with 0-10% survival scores

In [12]:
correct_circuit = load_program(ORIGINAL_QASM, ORIGINAL_QASM_FOLDER).copy()
correct_list = LinkedList()
correct_list = construct_list(correct_list, correct_circuit, inverse=False)

<h2 style="text-align: center;"> SB-QOPS </h2>

This is where we will use SB-QOPS on a provided circuit and its mutants

For a single mutant, it took about 2 minutes to generate a 20 test suite (10 fail, 10 pass) on an RTX 4090 SUPER. It can only be run on a Linux environment since the AER Simulator does not support Windows

There are 113 mutants for the DJ algorithm in use: 226 minutes or 3 3/4 hours

There are 89 mutants for the GHZ algorithm in use: 178 minutes or about 3 hours

There are 125 mutants for the RealAmpRandom algorithm in use: 250 minutes or just over 4 hours

BUT this cell should only need to be run once per algorithm with mutants under test. After it saves to the csv file at the end, this cell can be commented out as to not accidentally run it again. 

In the future if we want to test this methodology on higher qubit circuits or other algorithms, we'll likely either want to reduce the number of tests in the suite or the number of mutants we're analyzing

In [13]:
# import QOPS as qops
# from QOPS_test import get_compact_program_specification_Z
# from pathlib import Path

# ga_result = pd.DataFrame(columns=['Program',"path_to_mutant",'catch_avg','avg_fitness','failing_testcases', 'passing_testcases'])
# program_history = {}

# #program_specification = The compact program specification. SB-QOPS needs this to generate failing test cases.
# # In a public-use environment, this would be provided by the user.
# program_specification = get_compact_program_specification_Z(correct_circuit, shots=SHOTS)

# #mutants = A python list of qiskit QuantumCircuit variables with a mutation in them
# mutants = []
# mutants_names = []
# root = Path(QASM_MUTANT_FOLDER)
# for qasm_mutant in root.rglob("*.qasm"):
#     mutants.append(load_program(qasm_mutant.name, QASM_MUTANT_FOLDER))
#     mutants_names.append(qasm_mutant.name)

# for mutant_index,mutant in enumerate(mutants):
#     tester = qops.Circuit_Tester(CUT=mutant)
#     tester.set_applicable_families_Z(program_specification)
#     mutants_per_run = []
#     fitness_per_run = []
#     failing_testcases_per_run = []
#     history_per_run = []

#     print("NOW RUNNING TO FIND FAILING TESTS")
#     #For a predefined number of attempts, try to find a set of failing test cases (output is above predefined tolerance)
#     for runs in range(RUNS):
#         killed = 0
#         pauli = {}
#         fitness = 0
#         for i in range(len(tester.applicable_families)):
#             best_function,testcase, history = tester.run_mealoneplusone(i, EPOCH)
#             if best_function > SBQOPS_TOL:
#                 killed = 1
#                 pauli = testcase
#                 fitness = best_function
#                 break
#         mutants_per_run.append(killed)
#         failing_testcases_per_run.append(pauli)
#         fitness_per_run.append(fitness)
#         history_per_run.append(history)

#     avg_score = np.mean(mutants_per_run)
#     avg_fitness = np.mean(fitness_per_run)

#     #Here is our new, modified algorithm from the SB-QOPS method that attempts to find passing test cases as well. We'll need them for SBFL
#     passing_testcases_per_run = []

#     print("NOW RUNNING TO FIND PASSING TESTS")
#     for runs in range(RUNS):
#         pauli = {}
#         fitness = 0
#         for i in range(len(tester.applicable_families)):
#             best_function, testcase, history = tester.run_reverse_mealoneplusone(i,EPOCH)
#             if best_function < SBQOPS_TOL:
#                 pauli = testcase
#                 break
#         passing_testcases_per_run.append(pauli)

#     #Replace static program file with "program_name" when ready to test fully
#     """
#     ga_result: A pandas dataframe with the following columns
#         Program: Name of the mutant quantum circuit file
#         Path_To_Mutant: Path to the mutant file
#         Catch_Avg: The average number of times the mutant was identified by SB-QOPS
#         Avg_Fitness: The average fitness of the search algorithm used. Higher usually indicates higher Catch_Avg
#         Failing_Testcases: A list of dicts indicating the set of Pauli strings and the weights that should be applied to generate a failing test case
#         Passing_Testcases: A list of dicts indicating the set of Pauli strings and the weights that should be applied to generate a passing test case
#     """
#     ga_result = pd.concat([pd.DataFrame([[mutants_names[mutant_index],QASM_MUTANT_FOLDER,avg_score,avg_fitness,json.dumps(failing_testcases_per_run), json.dumps(passing_testcases_per_run)]],columns=ga_result.columns),ga_result],ignore_index=True)
#     ga_result.to_csv(f'realamprandom_ga_result', index=False)

ga_result is a slightly altered pandas Dataframe with similar structure found in SB-QOPS's results folder. The differences are changing the column "Program" to be the name of the mutant file instead of the original quantum circuit, changing "Mutant" to be the path to the mutant file instead of being an arbitrary index value, and adding a passing_testcases column that returns Pauli strings and coefficients that generate passing tests.

Now we want to run SBFL using Heisenberg evolution trees on each circuit placed in ga_result

Evolution analysis tends to take about 5 minutes for 10 failing tests on a very complex algorithm such as QNN, so about 10 minutes for 20 total tests

DJ was a relatively small algorithm with few to no branching gates, so it only ended up taking about 5-6 minutes to evolve and analyze all 113 DJ mutants

About 890 minutes for GHZ, or 14.8 hours to evolve and analyze all 89 GHZ mutants

About 1250 minutes for RealAmpRandom, or 20.8 hours to evolve and analyze all 125 RealAmpRandom mutants

The runtime of this code segment is directly tied to the depth of the circuit being analyzed and the number of 2-branching gates such as rotation or phase gates that exist in the circuit.

This is something to note in the final paper. Regardless of accuracy this methodology takes a long time to run. If results are promising, then we'll want to look into the tradeoffs between EXAM and hyperparameters to speed this up. Primarily the atol, max_terms, and search_step parameters in the evolve_pauli_exact method in HeisenbergUtilities.py


<h2 style="text-align: center;">HEISENBERG EVOLUTIONS (PAULI PROPAGATION)</h2>

In [14]:
from heisenbergTree import *

ga_result = pd.read_csv(f'{ALG_NAME}_ga_result.csv')
tarantula_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','exam_score'])
barinel_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','exam_score'])
mutant_num = 0

#For each mutant of a circuit, run the SBFL implementation
for mutant, path in zip(ga_result.loc[:,"Program"], ga_result.loc[:,"path_to_mutant"]):
    print("Now evolving the following mutant: ", mutant)

    #Extract the raw test cases found for each mutant
    desired_failing_testcases = ga_result.loc[(ga_result["Program"] == mutant), ["failing_testcases"]]
    desired_passing_testcases = ga_result.loc[(ga_result["Program"] == mutant), ["passing_testcases"]]

    dict_failing_testcases = desired_failing_testcases.to_dict()
    dict_passing_testcases = desired_passing_testcases.to_dict()

    string_failing_testcases = dict_failing_testcases['failing_testcases'][mutant_num]
    string_passing_testcases = dict_passing_testcases['passing_testcases'][mutant_num]

    list_failing_testcases = json.loads(string_failing_testcases)
    list_passing_testcases = json.loads(string_passing_testcases)

    #Remove empty tests
    raw_failing_testcases = remove_null_tests(list_failing_testcases)
    raw_passing_testcases = remove_null_tests(list_passing_testcases)

    circuit_inverse = load_program(mutant,path).copy().inverse()  # Invert to track backward evolution of the operator
    forward_mutant = load_program(mutant, path).copy()

    #Create the linked list of operations in the inverse circuit
    operation_list = LinkedList()
    operation_list = construct_list(operation_list, circuit_inverse, True)

    forward_list = LinkedList()
    forward_list = construct_list(forward_list, forward_mutant, False)

    #Create list of nodes in linked list. Somewhere down the road remove the linked list implementation. It's unnecessary and giving me a headache
    node_list = []
    node = operation_list.head
    while node:
        node_list.append(node)
        node = node.next

    #Perform Pauli propagation (Heisenberg evolution) for all test cases. Returns a dataframe with all counts for all operations
    global_frame_fail = heisenberg_evolve(circuit_inverse, 
                                          operation_list, 
                                          raw_failing_testcases, 
                                          "fail", 
                                          LAMBDA_PHASE, 
                                          LAMBDA_CHANGE, 
                                          LAMBDA_SIMILARITY, 
                                          atol = ATOL, 
                                          max_terms = MAX_TERMS, 
                                          search_step = SEARCH_STEP)
    
    global_frame_pass = heisenberg_evolve(circuit_inverse, 
                                          operation_list, 
                                          raw_passing_testcases, 
                                          "pass", 
                                          LAMBDA_PHASE, 
                                          LAMBDA_CHANGE, 
                                          LAMBDA_SIMILARITY, 
                                          atol = ATOL, 
                                          max_terms = MAX_TERMS, 
                                          search_step = SEARCH_STEP)

    global_frame = pd.concat([global_frame_fail, global_frame_pass], ignore_index=True)

    global_frame = global_frame.drop(["Initial"],axis=1)
    if VERBOSE:
        display(global_frame)

    tarantula_scores = tarantula(global_frame)
    barinel_scores = barinel(global_frame)

    if VERBOSE:
        print("BARINEL SCORES")
        display(barinel_scores)
        print("TARANTULA SCORES")
        display(tarantula_scores)
    
    # Here is where analysis of the methodology begins. 
    # We first extract the position of the erroneous gate by comparing it to the original MQT gate
    # NOTE: This method is intended for single-added gates for now. Other types of mutants will require other methods later
    error_gate_location = find_erroneous_gate(forward_mutant, correct_circuit)

    if VERBOSE:
        print("ERROR GATE LOCATION:")
        print(error_gate_location)

    # Calculate the EXAM score for Barinel by counting the number of gates we would have to observe before we reach the erroneous gate.
    gates_observed = 0
    barinel_exam_score = 0
    for col_name, col_date in barinel_scores.items():
        gates_observed += 1

        #Extract depth from column name
        gate_depth = int(col_name.split()[1])

        if gate_depth == error_gate_location:
            barinel_exam_score = gates_observed/(circuit_inverse.size())
            break

    # Calculate the EXAM score for Tarantula by counting the number of gates we would have to observe before we reach the erroneous gate.
    gates_observed = 0
    tarantula_exam_score = 0
    for col_name, col_date in tarantula_scores.items():
        gates_observed += 1

        #Extract depth from column name
        gate_depth = int(col_name.split()[1])

        if gate_depth == error_gate_location:
            tarantula_exam_score = gates_observed/(circuit_inverse.size())
            break

    # Here we store the results from the HBFL analysis for both barinel and tarantula into a csv file for later analysis.
    barinel_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,barinel_exam_score]],columns=barinel_sbfl_result.columns),barinel_sbfl_result],ignore_index=True)
    barinel_sbfl_result.sort_values(by=['exam_score'], ascending=False)
    barinel_sbfl_result.to_csv(f'{ALG_NAME}_barinel_sbfl_result.csv', index=False)

    tarantula_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,tarantula_exam_score]],columns=tarantula_sbfl_result.columns),tarantula_sbfl_result],ignore_index=True)
    tarantula_sbfl_result.sort_values(by=['exam_score'], ascending=False)
    tarantula_sbfl_result.to_csv(f'{ALG_NAME}_tarantula_sbfl_result.csv', index=False)

    mutant_num += 1

if VERBOSE:
    display(barinel_sbfl_result)
    display(tarantula_sbfl_result)
    

Now evolving the following mutant:  AddGate_y_inGap_9_.qasm


  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:41<00:00,  4.14s/it]


,h 21,cp 20,cp 19,cp 18,h 17,cp 16,cp 15,h 14,y 13,cp 12,...,swap 8,cp 7,cp 6,cp 5,x 4,h 3,h 2,h 1,h 0,pass/fail
0,0.539234,0.791537,0.401934,0.111636,0.760402,1.017375,0.488838,0.977856,0.355861,1.226513,...,0.387754,1.078502,0.300790,1.156294,0.152645,0.495605,0.293450,0.258724,0.263727,fail
1,0.586147,0.871685,0.418678,0.121474,0.698199,0.961836,0.457415,0.887303,0.334486,1.161036,...,0.340350,1.007323,0.277582,1.185145,0.145437,0.421422,0.268147,0.238052,0.258499,fail
2,0.472807,0.734793,0.316530,0.092475,0.599763,0.747743,0.355460,0.682064,0.270455,0.895294,...,0.313248,0.806797,0.215508,0.912850,0.115231,0.344795,0.207687,0.189896,0.201115,fail
3,0.504371,0.735491,0.364344,0.107342,0.648379,0.885665,0.429787,0.871451,0.329923,1.142263,...,0.314227,0.929933,0.250460,1.056675,0.143818,0.427162,0.258149,0.219827,0.234315,fail
4,0.432447,0.654748,0.298943,0.098716,0.576863,0.763969,0.385656,0.709691,0.261371,0.935523,...,0.289235,0.843344,0.239736,0.936957,0.116851,0.425163,0.227242,0.208937,0.213006,fail
5,0.605406,0.877627,0.412200,0.125801,0.726220,1.014731,0.485417,0.837148,0.329255,1.125963,...,0.327589,0.978322,0.294748,1.178977,0.152918,0.423211,0.257963,0.246852,0.255088,fail
6,0.615256,0.921742,0.395683,0.130843,0.676089,0.931106,0.429381,0.692124,0.265414,0.935276,...,0.207725,0.762909,0.248028,1.093910,0.129434,0.376548,0.213405,0.223205,0.238793,fail
7,0.636323,0.913075,0.422326,0.134188,0.858247,1.138007,0.551507,0.997655,0.357717,1.246209,...,0.340076,1.103151,0.346629,1.311917,0.160662,0.568685,0.311847,0.297509,0.298932,fail
8,0.388406,0.568526,0.270751,0.086752,0.489046,0.699970,0.337203,0.636689,0.258765,0.905054,...,0.272678,0.774920,0.193964,0.825845,0.108964,0.350883,0.204592,0.173543,0.186263,fail
9,0.527898,0.804508,0.361309,0.109378,0.722811,0.939093,0.476183,0.902636,0.346323,1.186340,...,0.390863,1.026363,0.296756,1.078834,0.141452,0.459972,0.273251,0.246718,0.240065,fail


BARINEL SCORES


...

TARANTULA SCORES


,rzz 16,swap 8,cp 6,cp 14,cp 15,swap 10,cp 12,cp 9,h 13,cp 7,...,x 4,h 1,h 2,h 3,cp 5,cp 19,h 0,h 21,cp 18,cp 20
sum,0.552117,0.543782,0.54224,0.541578,0.536888,0.531156,0.530267,0.529748,0.52866,0.528125,...,0.521929,0.520068,0.517303,0.516885,0.498831,0.497638,0.497453,0.495077,0.492574,0.49025


ERROR GATE LOCATION:
16


,Program,path_to_mutant,exam_score
0,AddGate_rzz_inGap_10_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.045455
1,AddGate_z_inGap_6_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.136364
2,AddGate_z_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.090909
3,AddGate_swap_inGap_10_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.045455
4,AddGate_y_inGap_14_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.045455
5,AddGate_x_inGap_1_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.181818
6,AddGate_y_inGap_11_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.272727
7,AddGate_x_inGap_2_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.363636
8,AddGate_z_inGap_7_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.5
9,AddGate_p_inGap_11_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.181818


,Program,path_to_mutant,exam_score
0,AddGate_rzz_inGap_10_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.045455
1,AddGate_z_inGap_6_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.136364
2,AddGate_z_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.090909
3,AddGate_swap_inGap_10_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.045455
4,AddGate_y_inGap_14_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.045455
5,AddGate_x_inGap_1_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.181818
6,AddGate_y_inGap_11_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.272727
7,AddGate_x_inGap_2_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.363636
8,AddGate_z_inGap_7_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.5
9,AddGate_p_inGap_11_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_qpeexac...,0.227273
